In [ ]:
#@title 1. Install compatible dependencies
# Run once per fresh Colab runtime. Do NOT upgrade torch or install torchao separately.
import sys
import subprocess
import os
import urllib.request

packages = [
    "diffusers>=0.30.0",
    "transformers>=4.44.0",
    "accelerate>=0.33.0",
    "safetensors",
    "fastapi",
    "uvicorn",
    "pyngrok",
    "pillow",
    "pydantic>=2.0",
]

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", *packages
])

# Install Cloudflare quick tunnel binary (temporary URL; changes on each runtime).
cloudflared_path = "/usr/local/bin/cloudflared"
try:
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        cloudflared_path,
    )
    os.chmod(cloudflared_path, 0o755)
    print("Cloudflare Tunnel installed.")
except Exception as exc:
    print(f"Cloudflare Tunnel install failed: {exc}")
    print("Choose Ngrok in cell 3 and enter a valid Ngrok authtoken.")

print("Dependencies installed.")
print("If pip reports that packages were changed, restart the runtime before running cell 2.")


In [ ]:
#@title 2. Select model and load GPU pipeline
import torch
from diffusers import AutoPipelineForImage2Image

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. In Colab select Runtime > Change runtime type > T4 GPU, "
        "then run the cells again."
    )

MODEL_CHOICE = "stabilityai/sdxl-turbo" #@param ["stabilityai/sdxl-turbo", "stabilityai/sd-turbo"]

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GiB")
print(f"Loading {MODEL_CHOICE}...")

try:
    pipe = AutoPipelineForImage2Image.from_pretrained(
        MODEL_CHOICE,
        torch_dtype=torch.float16,
        variant="fp16",
        use_safetensors=True,
    )
    pipe = pipe.to("cuda")
    pipe.enable_attention_slicing()
    print("Model loaded successfully.")
except Exception as exc:
    print(f"Model loading failed: {type(exc).__name__}: {exc}")
    print("Do not install/upgrade torchao to fix this automatically. Check the full traceback.")
    raise


In [ ]:
#@title 3. Start FastAPI inference server and public tunnel
import base64
import datetime
import io
import re
import secrets
import shutil
import subprocess
import threading
import time
import traceback
import urllib.request
from typing import Optional

import torch
import uvicorn
from fastapi import FastAPI, Header, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from PIL import Image, UnidentifiedImageError

# Enter a long random token and use the SAME token in your backend .env.
# If left empty, a token is generated for this session; copy it to .env every time.
AUTH_TOKEN = "" #@param {type:"string"}
TUNNEL_PROVIDER = "Cloudflare" #@param ["Cloudflare", "Ngrok"]
NGROK_AUTHTOKEN = "" #@param {type:"string"}
PORT = 7860

if not AUTH_TOKEN.strip():
    AUTH_TOKEN = secrets.token_urlsafe(32)
    print("Generated a temporary auth token for this runtime.")

class GenerationRequest(BaseModel):
    image_base64: str = Field(min_length=1, max_length=20_000_000)
    prompt: str = Field(min_length=1, max_length=5000)
    strength: float = Field(default=0.65, ge=0.05, le=1.0)
    steps: int = Field(default=4, ge=1, le=10)

app = FastAPI(title="AI Room Makeover Inference Server")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=False,
    allow_methods=["GET", "POST"],
    allow_headers=["Authorization", "Content-Type"],
)
generation_lock = threading.Lock()

@app.get("/health")
def health():
    return {"status": "ok", "gpu": torch.cuda.get_device_name(0), "model": MODEL_CHOICE}

@app.post("/api/try-on")
def generate(request: GenerationRequest, authorization: Optional[str] = Header(default=None)):
    if AUTH_TOKEN.strip() and (
        authorization is None
        or not secrets.compare_digest(authorization.strip(), f"Bearer {AUTH_TOKEN.strip()}")
    ):
        raise HTTPException(status_code=401, detail="Invalid inference token")

    started = time.time()
    try:
        raw_b64 = request.image_base64.strip()
        if raw_b64.lower().startswith("data:") and "," in raw_b64:
            raw_b64 = raw_b64.split(",", 1)[1]
        try:
            image_bytes = base64.b64decode(raw_b64, validate=True)
        except Exception as exc:
            raise HTTPException(status_code=400, detail="image_base64 is not valid Base64") from exc

        try:
            with Image.open(io.BytesIO(image_bytes)) as source:
                image = source.convert("RGB")
        except (UnidentifiedImageError, OSError) as exc:
            raise HTTPException(status_code=400, detail="Uploaded data is not a valid image") from exc

        max_dim = 1024 if "sdxl" in MODEL_CHOICE.lower() else 512
        image.thumbnail((max_dim, max_dim), Image.Resampling.LANCZOS)
        width = max(64, (image.width // 8) * 8)
        height = max(64, (image.height // 8) * 8)
        image = image.resize((width, height), Image.Resampling.LANCZOS)

        steps = min(request.steps, 4)
        with generation_lock, torch.inference_mode():
            result = pipe(
                prompt=request.prompt,
                image=image,
                strength=request.strength,
                num_inference_steps=steps,
                guidance_scale=0.0,
            ).images[0]

        buffer = io.BytesIO()
        result.save(buffer, format="PNG")
        encoded = base64.b64encode(buffer.getvalue()).decode("ascii")
        print(f"[{datetime.datetime.now().isoformat(timespec='seconds')}] Generated {result.size} in {time.time()-started:.1f}s")
        return {"image_base64": encoded}
    except HTTPException:
        raise
    except Exception as exc:
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=f"Image generation failed: {exc}") from exc
    finally:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

# Start API server. If this port is already in use, restart the Colab runtime and run cells from the top.
server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=PORT, log_level="info"))
server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()

local_health = f"http://127.0.0.1:{PORT}/health"
for _ in range(60):
    try:
        with urllib.request.urlopen(local_health, timeout=2) as response:
            if response.status == 200:
                break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("FastAPI did not start. Restart the Colab runtime and rerun cells from the top.")

public_url = None
tunnel_process = None
tunnel_lines = []

if TUNNEL_PROVIDER == "Cloudflare":
    cloudflared = shutil.which("cloudflared")
    if not cloudflared:
        raise RuntimeError("cloudflared is not installed. Rerun cell 1 or choose Ngrok.")
    tunnel_process = subprocess.Popen(
        [cloudflared, "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )

    def read_tunnel_logs():
        global public_url
        if tunnel_process.stdout:
            for line in iter(tunnel_process.stdout.readline, ""):
                tunnel_lines.append(line.rstrip())
                match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
                if match:
                    public_url = match.group(0)

    threading.Thread(target=read_tunnel_logs, daemon=True).start()
    for _ in range(60):
        if public_url:
            break
        if tunnel_process.poll() is not None:
            break
        time.sleep(1)
else:
    if not NGROK_AUTHTOKEN.strip():
        raise RuntimeError("Enter a valid NGROK_AUTHTOKEN or select Cloudflare.")
    from pyngrok import ngrok
    ngrok.kill()
    ngrok.set_auth_token(NGROK_AUTHTOKEN.strip())
    tunnel = ngrok.connect(PORT, "http")
    public_url = tunnel.public_url

if not public_url:
    raise RuntimeError(
        "Could not obtain a public tunnel URL. Try Ngrok or rerun the tunnel cell.\n"
        + "\n".join(tunnel_lines[-15:])
    )

endpoint_url = f"{public_url}/api/try-on"
print("\nAI ROOM MAKEOVER SERVER IS READY")
print(f"COLAB_INFERENCE_URL={endpoint_url}")
print(f"COLAB_INFERENCE_TOKEN={AUTH_TOKEN}")
print(f"Health check: {public_url}/health")
print("Copy both URL and token into backend/.env.")
print("This temporary URL and GPU session stop when Colab disconnects.")
